# Smart Bundle Recommendation (Telecom)

This notebook builds a telecom bundle recommendation model using the Kaggle **Churn in Telecoms** dataset.

**Approach:** Unsupervised clustering (K-Means) on usage profiles, then map clusters to bundle tiers.

**Goal:** Recommend voice/data bundles based on historical usage patterns.

**Note:** The dataset does not include direct mobile data usage, so we create a *data proxy* feature from evening and night usage. This is documented as a limitation.


## 1. Load Data
Place the Kaggle CSV in `data/telecom_churn.csv`.

In [ ]:
import pandas as pd
from pathlib import Path

data_path = Path('../data/telecom_churn.csv')
df_raw = pd.read_csv(data_path)
df_raw.head()

In [ ]:
df_raw.shape

## 2. Initial EDA and Data Quality
We check missing values, duplicates, and data types.

In [ ]:
df_raw.info()

In [ ]:
df_raw.isna().sum().sort_values(ascending=False).head(10)

In [ ]:
df_raw.duplicated().sum()

## 3. Cleaning + Feature Engineering
We standardize columns, convert Yes/No to 1/0, fill missing values,
then create usage features including `Total minutes` and a `Data proxy`.

In [ ]:
import sys
sys.path.append('../src')
from features import build_training_frame, model_features

df = build_training_frame(df_raw)
df[['Total minutes', 'Data proxy', 'Intl usage ratio']].describe()

## 4. Modeling Approach: K-Means Clustering
We use K-Means to group users into usage clusters.
The best number of clusters is chosen with the Silhouette Score.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

X = model_features(df)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

scores = {}
for k in range(3, 8):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    scores[k] = silhouette_score(X_scaled, labels)

scores

In [ ]:
best_k = max(scores, key=scores.get)
best_k

In [ ]:
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
labels = kmeans.fit_predict(X_scaled)
df['Cluster'] = labels
df['Cluster'].value_counts()

## 5. Cluster Profiles and Interpretation
We summarize each cluster and map it to a bundle tier for recommendations.

In [ ]:
cluster_profiles = df.groupby('Cluster')[
    ['Total minutes', 'Data proxy', 'Intl usage ratio', 'Total intl minutes', 'Total calls']
].mean().round(2)
cluster_profiles

## 6. Save the Model
We persist the scaler + k-means model for the Streamlit UI.

In [ ]:
import joblib

model_bundle = {'scaler': scaler, 'kmeans': kmeans}
joblib.dump(model_bundle, '../models/bundle_model.pkl')